# fhkit.at: what a visitor gets

Checks on the built `public/` for the improvement pass of 2026-10-10: the
navigation signs in through /login/ and offers no registration, the PhotonQ
leftovers (docs placeholder, blog, social client, search) are gone, unknown
addresses get a 404 page in their language, dark mode has readable colours,
the server HTML nests nothing a browser would rewrite, and the four foreign
locales show translated texts, stored as CMS content as well.

Run after `gatsby build`. Nothing here talks to the network; the catalog
check runs the repository's own esbuild and node.

In [ ]:
import html, json, os, re, subprocess
from html.parser import HTMLParser
from pathlib import Path

SITE = Path(os.environ.get('FHKIT_SITE_DIR', '..')).resolve()
PUBLIC = SITE / 'public'
LOCALES = ['de', 'en', 'sl', 'it', 'ja']
DEFAULT = 'de'
PREFIX = {loc: '' if loc == DEFAULT else f'{loc}/' for loc in LOCALES}

RESULTS = []

def check(name, ok, detail=''):
    RESULTS.append({'name': name, 'pass': bool(ok), 'detail': str(detail)})
    print(('PASS  ' if ok else 'FAIL  ') + name + (f'  ({detail})' if detail else ''))
    return bool(ok)

def read(rel):
    return (PUBLIC / rel).read_text(encoding='utf8')

def text_of(source):
    s = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', source, flags=re.S)
    s = s.replace('<!-- -->', '')
    s = re.sub(r'<[^>]+>', ' ', s)
    return re.sub(r'\s+', ' ', html.unescape(s))

# Files only: the 404 page's data lives in a directory named page-data/404.html.
PAGES = sorted(p for p in PUBLIC.rglob('*.html') if p.is_file())
SITE_PAGES = [p for p in PAGES if not any(f'/{s}/' in '/' + str(p.relative_to(PUBLIC))
              for s in ['cms', 'login', 'logout', 'settings', 'signup', 'emailwerk'])]
TEXT_FILES = [p for p in PUBLIC.rglob('*') if p.is_file()
              and p.suffix in ('.html', '.js', '.json', '.txt', '.xml')]
CONTENT = {p: p.read_text(encoding='utf8', errors='ignore') for p in TEXT_FILES}

def files_with(needle, files=None):
    return sorted(str(p.relative_to(PUBLIC)) for p in (files or CONTENT) if needle in CONTENT[p])

check('public/ exists and is not empty', PUBLIC.is_dir() and PAGES, f'{len(PAGES)} html files')

## Navigation

Sign In is a plain link to /login/, which the browser follows with a full
page load. Calling `signinRedirect` from the bar threw "the OIDC runtime is
not loaded on this route", because jaen loads its sign-in code only on its
sign-in routes. There is no Sign Up: fhkit.at has no visitor accounts. The
labels and the targets follow the page's language.

In [ ]:
SIGN_IN = {'de': 'Anmelden', 'en': 'Sign in', 'sl': 'Prijava', 'it': 'Accedi', 'ja': 'ログイン'}
HOME = {'de': 'Startseite', 'en': 'Home', 'sl': 'Domov', 'it': 'Home', 'ja': 'ホーム'}

def anchors(source):
    return re.findall(r'<a\b([^>]*)>(.*?)</a>', source, flags=re.S)

for loc in LOCALES:
    source = read(f'{PREFIX[loc]}imprint/index.html')
    nav = source[source.find('<nav'):source.find('</nav>')]
    links = [(attrs, text_of(inner).strip()) for attrs, inner in anchors(nav)]
    login = [t for a, t in links if 'href="/login/"' in a]
    check(f'{loc}: the bar links {SIGN_IN[loc]!r} to /login/', login == [SIGN_IN[loc]], login)
    home = f'href="/{PREFIX[loc]}"'
    check(f'{loc}: the bar links {HOME[loc]!r} to its own home page',
          any(home in a and t == HOME[loc] for a, t in links), [t for a, t in links])
    check(f'{loc}: the bar links the legal notice of its locale',
          any(f'href="/{PREFIX[loc]}imprint/"' in a for a, t in links))
    check(f'{loc}: every link in the bar has a target',
          all('href=' in a for a, t in links), [t for a, t in links if 'href=' not in a])

for needle in ['Sign Up', 'Sign up', 'Registrieren']:
    check(f'no page offers {needle!r}', not files_with(needle, SITE_PAGES))

## Template leftovers

The docs pages only ever showed jaen's "Hello, world!" placeholder, the blog
was PhotonQ's community page fed by PhotonQ's social pylon, and the search
had nothing to find. All three are gone from the build, from the links and
from the sitemap. So is the home bar's hidden menu with the old address
Löwengasse 28.

In [ ]:
routes = {'/' + p.parent.relative_to(PUBLIC).as_posix() for p in PUBLIC.rglob('index.html')}
for loc in LOCALES:
    for gone in ['docs', 'experiments']:
        route = f'/{PREFIX[loc]}{gone}'
        check(f'{route}/ is not built', not any(r == route or r.startswith(route + '/') for r in routes))

linked = sorted({str(p.relative_to(PUBLIC)) for p in SITE_PAGES
                 if re.search(r'href="(/[a-z]{2})?/(docs|experiments)\b', CONTENT[p])})
check('no page links to /docs or /experiments', not linked, linked[:5])

sitemap = read('sitemap.xml')
locs = sorted(re.findall(r'<loc>([^<]+)</loc>', sitemap))
expected = sorted(f'https://fhkit.at/{PREFIX[l]}{r}' for l in LOCALES for r in ['', 'imprint/'])
check('sitemap.xml lists the home page and the legal notice of every locale, nothing else',
      locs == expected, locs)

for needle, files in [('Hello, world!', SITE_PAGES), ('Löwengasse 28', SITE_PAGES),
                      ('Unsere Services', SITE_PAGES), ('to search', SITE_PAGES),
                      ('pylons.photonq.org', None), ('OpenQASM', None)]:
    hits = files_with(needle, files)
    check(f'no {needle!r} in {"any page" if files else "any built file"}', not hits, hits[:5])
check('no search-index.json', not (PUBLIC / 'search-index.json').exists())

## 404

Unknown addresses used to get the home page with status 200, because without
a 404.html Cloudflare Pages treats a site as a single-page app. Gatsby writes
/404.html and gatsby-node adds one per locale; Pages serves the nearest one
with status 404. The browser renders the page data of /404.html everywhere,
so every 404 page carries `notFound` and takes its language from the
address.

In [ ]:
NOT_FOUND = {'de': 'Seite nicht gefunden', 'en': 'Page not found', 'sl': 'Strani ni mogoče najti',
             'it': 'Pagina non trovata', 'ja': 'ページが見つかりません'}

for loc in LOCALES:
    rel = f'{PREFIX[loc]}404.html'
    if not check(f'{rel} exists', (PUBLIC / rel).is_file()):
        continue
    source = read(rel)
    lang = re.search(r'<html[^>]*lang="([^"]+)"', source)
    check(f'{rel}: html lang is {loc}', lang and lang.group(1) == loc, lang and lang.group(1))
    title = re.search(r'<title[^>]*>([^<]*)', source)
    check(f'{rel}: title {NOT_FOUND[loc]!r}', title and title.group(1).strip() == NOT_FOUND[loc],
          title and title.group(1))
    check(f'{rel}: heading {NOT_FOUND[loc]!r}',
          re.search(r'<h1[^>]*>' + re.escape(NOT_FOUND[loc]) + '</h1>', source) is not None)
    check(f'{rel}: robots noindex', re.search(r'<meta name="robots" content="noindex"', source) is not None)
    check(f'{rel}: links to /{PREFIX[loc]}', f'href="/{PREFIX[loc]}"' in source)

context = json.loads(read('page-data/404.html/page-data.json'))['result']['pageContext']
check('the page data of /404.html carries notFound', context.get('notFound') is True, context.get('notFound'))
check('no 404 page in the sitemap', '404' not in sitemap)

## Dark mode

The brand palette's dark halves were steps of the near-black brand ramp, so
in dark mode solid buttons were dark labels on a dark field. The hero heading
was hard-coded #000000, the social icons near-black, and the bar's current
page black at 48% opacity. Read off the CSS that the home page ships.

In [ ]:
home = read('index.html')

def rules_for(cls, source=home):
    out, start = [], 0
    while True:
        i = source.find('.' + cls, start)
        if i < 0:
            return out
        a, b = source.rfind('}', 0, i) + 1, source.find('}', i) + 1
        out.append(source[a:b])
        start = i + 1

def dark_color(cls):
    return any(r.lstrip().startswith('.dark ') and 'color:' in r for r in rules_for(cls))

i = home.find('--chakra-colors-brand-solid:var(--chakra-colors-gray-100)')
block = home[home.rfind('{', 0, i):home.find('}', i)] if i >= 0 else ''
selector = home[home.rfind('}', 0, i) + 1:home.rfind('{', 0, i)] if i >= 0 else ''
check('dark mode: the brand palette is defined for .dark', selector.startswith('.dark'), selector[:40])
for slot, value in [('solid', 'gray-100'), ('contrast', 'gray-900'), ('fg', 'gray-200')]:
    check(f'dark mode: brand.{slot} is {value}', f'--chakra-colors-brand-{slot}:var(--chakra-colors-{value})' in block)
ramp = re.findall(r'--chakra-colors-brand-[a-z-]+:var\(--chakra-colors-brand-\d+\)', block)
check('dark mode: no palette slot points at the near-black ramp', not ramp, ramp)
check('dark mode: the active top-bar entry is white',
      re.search(r'--chakra-colors-top-nav-links-active-color:(white|var\(--chakra-colors-white\))', block)
      is not None)
check('html carries the page colour too',
      re.search(r'html\{[^}]*background:var\(--chakra-colors-shared-body-bg-color\)', home) is not None)

heading = re.search(r'<h2 class="([^"]*)"[^>]*id="HeroSectionHeading"', home)
classes = heading.group(1).split() if heading else []
check('the hero heading turns light in dark mode',
      any(dark_color(c) for c in classes if c.startswith('css-')), classes)

def classes_of(tag):
    m = re.search(r'class="([^"]*)"', tag or '')
    return [c for c in (m.group(1).split() if m else []) if c.startswith('css-')]

for label in ['Netsnek', 'Github', 'Linkedin', 'Instagram', 'Telegram']:
    tag = re.search(r'<a\b[^>]*aria-label="' + label + '"[^>]*>', home)
    cls = classes_of(tag and tag.group(0))
    check(f'the {label} icon turns light in dark mode', any(dark_color(c) for c in cls), cls)

button = re.search(r'<button[^>]*class="([^"]*)"[^>]*>Kontaktiere mich</button>', home)
cls = [c for c in (button.group(1).split() if button else []) if c.startswith('css-')]
forced = [r for c in cls for r in rules_for(c) if 'color:var(--chakra-colors-white)' in r]
check('"Kontaktiere mich" takes the palette contrast colour instead of forced white',
      button and not forced, forced[:2])

## Partner cards

The three partner cards had no flex basis, so each took the width of its own
text and the Kanbon card was squeezed to about 165px. From md up they share
the row equally.

In [ ]:
wrappers = re.findall(r'<div class="(css-[a-z0-9]+)">(?:<style[^>]*>[^<]*</style>)?'
                      r'<div class="chakra-stack css-[a-z0-9]+">(?:<style[^>]*>[^<]*</style>)?<h3', home)
check('the three partner cards share one wrapper style', len(wrappers) == 3 and len(set(wrappers)) == 1, wrappers)
rules = rules_for(wrappers[0]) if wrappers else []
check('from md up each card takes an equal share of the row',
      any('min-width: 48rem' in r and 'flex:1;' in r for r in rules), rules)
check('a card may shrink below its longest word', any('min-width:0' in r for r in rules), rules)

## Markup the browser would rewrite

A `<p>` inside a `<p>` is invalid HTML: the parser closes the outer one early
and the DOM no longer matches what React rendered. That is what threw
hydration errors #418 and #423 on every home page, from the partner cards,
whose CMS text fields render their own paragraph.

In [ ]:
BLOCKS = {'address', 'article', 'aside', 'blockquote', 'details', 'div', 'dl', 'fieldset', 'figcaption',
          'figure', 'footer', 'form', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'header', 'hr', 'main', 'menu',
          'nav', 'ol', 'p', 'pre', 'section', 'table', 'ul'}
VOID = {'area', 'base', 'br', 'col', 'embed', 'hr', 'img', 'input', 'link', 'meta', 'source', 'track', 'wbr'}

class Nesting(HTMLParser):
    def __init__(self):
        super().__init__()
        self.stack, self.hits = [], []
    def handle_starttag(self, tag, attrs):
        if tag in BLOCKS and 'p' in self.stack:
            self.hits.append(f'<{tag} id={dict(attrs).get("id")}> inside <p>')
        if tag == 'a' and 'a' in self.stack:
            self.hits.append('<a> inside <a>')
        if tag not in VOID:
            self.stack.append(tag)
    def handle_endtag(self, tag):
        if tag in self.stack:
            del self.stack[len(self.stack) - 1 - self.stack[::-1].index(tag):]

def nesting_hits(source):
    parser = Nesting()
    parser.feed(source[source.find('<div id="___gatsby"'):])
    return parser.hits

for loc in LOCALES:
    hits = nesting_hits(read(f'{PREFIX[loc]}index.html'))
    check(f'{loc}: the home page nests no block element inside a paragraph', not hits, hits[:3])

bad = {str(p.relative_to(PUBLIC)): nesting_hits(CONTENT[p]) for p in SITE_PAGES}
bad = {page: hits for page, hits in bad.items() if hits}
check('no other site page does either', not bad, list(bad)[:5])

## Translations

/en/, /sl/, /it/ and /ja/ showed the German CMS texts under translated menus.
The texts' defaults come from the catalog now, and jaen-data/patch-i18n.json
stores the same translations as content of the localized pages. Each locale
must show its own texts and none of the German ones, the patch must be applied,
and it must match the catalog, so a text changed in the catalog without
rerunning `node jaen-data/build-patch-i18n.mjs` fails here.

In [ ]:
esbuild = SITE / 'node_modules' / 'esbuild' / 'bin' / 'esbuild'
bundle = subprocess.run([str(esbuild), str(SITE / 'src' / 'locales' / 'messages.ts'), '--bundle',
                         '--format=cjs', '--log-level=warning'], capture_output=True, text=True, check=True).stdout
dump = subprocess.run(['node', '-e', "let c='';process.stdin.on('data',d=>c+=d).on('end',()=>{"
                       "const m={exports:{}};new Function('module','exports',c)(m,m.exports);"
                       "process.stdout.write(JSON.stringify(m.exports.default))})"],
                      input=bundle, capture_output=True, text=True, check=True).stdout
CATALOG = json.loads(dump)
check('the catalog has the five locales', sorted(CATALOG) == sorted(LOCALES), sorted(CATALOG))

plain = lambda s: re.sub(r'\s+', ' ', re.sub(r'<br\s*/?>', ' ', s)).strip()
GERMAN = ['Lass dich von', 'Willkommen!', 'Es war schon immer', 'Danke für deinen Besuch', 'Für dich',
          'Kontaktiere mich', 'Software Entwicklung', 'Büroräume', 'Gestaltet von']
SHOWN = ['HeroSectionHeading', 'HeroBadgeText', 'HeroContactButton', 'ContentSectionHeading',
         'ContentSectionHeadingTeam', 'PartnerElectricalHeading', 'MapSectionHeading', 'FooterTitleLine1']

for loc in LOCALES:
    text = text_of(read(f'{PREFIX[loc]}index.html'))
    missing = [k for k in SHOWN if plain(CATALOG[loc][k]) not in text]
    check(f'{loc}: the home page shows its own texts', not missing, missing)
    if loc != DEFAULT:
        left = [g for g in GERMAN if g in text]
        check(f'{loc}: the home page shows no German text', not left, left)
        footer = text_of(read(f'{PREFIX[loc]}imprint/index.html'))
        check(f'{loc}: the legal notice has a translated footer',
              plain(CATALOG[loc]['FooterTitleLine1']) in footer and 'Danke für deinen Besuch' not in footer)

patches = [l.strip() for l in (SITE / 'jaen-data' / 'patches.txt').read_text().splitlines() if l.strip()]
check('patches.txt applies patch-i18n.json after the CMS patches',
      patches[-1] == 'patch-i18n.json' and all(l.startswith('http') for l in patches[:-1]), patches)

patch = json.loads((SITE / 'jaen-data' / 'patch-i18n.json').read_text(encoding='utf8'))
ids = sorted(p['id'] for p in patch['data']['pages'])
check('the patch covers the home page and the legal notice of each foreign locale',
      ids == sorted(f'JaenPage /{l}/{r}' for l in LOCALES if l != DEFAULT for r in ['', 'imprint/']), ids)

FOOTER_TITLES = {'FooterLinkTitleGestaltet von': 'FooterLinkTitleDesignedBy'}
stale, unapplied = [], []
for entry in patch['data']['pages']:
    loc = entry['id'].split('/')[1]
    fields = entry['jaenFields']['IMA:TextField']
    for name, field in fields.items():
        if field['value'] != CATALOG[loc][FOOTER_TITLES.get(name, name)]:
            stale.append(f'{entry["id"]} {name}')
    built_path = entry['id'][len('JaenPage /'):]
    built = json.loads(read(f'page-data/{built_path}page-data.json'))['result']['data']['jaenPage']
    built_fields = ((built or {}).get('jaenFields') or {}).get('IMA:TextField') or {}
    if any(built_fields.get(n, {}).get('value') != f['value'] for n, f in fields.items()):
        unapplied.append(entry['id'])
check('the patch matches the catalog', not stale, stale[:5])
check('the build carries the patch in the page data', not unapplied, unapplied)

## Google Maps placeholder

The notice that replaces the map until analytics cookies are allowed was
hard-coded German with a typo on every locale. It reads the catalog now.

In [ ]:
hits = files_with('Google Maps is nicht')
check('no "Google Maps is nicht verfügbar" in any built file', not hits, hits[:5])

In [ ]:
passed = sum(1 for r in RESULTS if r['pass'])
total = len(RESULTS)
print(f'\n{passed}/{total} checks passed')

out = SITE / 'tests' / 'results-fhkit-public-site.json'
out.write_text(json.dumps({'passed': passed, 'total': total, 'checks': RESULTS},
                          indent=1, ensure_ascii=False), encoding='utf8')
print('written to', out)

assert passed == total, f'{total - passed} check(s) failed'